In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")


In [3]:
df = pd.read_csv("heart.csv")

print("First Five Records")
print(df.head())

print("\nDataset Shape:", df.shape)

First Five Records
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak
0   63    1   3       145   233    1        0      150      0      2.3
1   37    1   2       130   250    0        1      187      0      3.5
2   41    0   1       130   204    0        0      172      0      1.4
3   56    1   1       120   236    0        1      178      0      0.8
4   57    0   0       120   354    0        1      163      1      0.6

Dataset Shape: (303, 10)


In [4]:
print("\nMissing Values")
print(df.isnull().sum())

print("\nDataset Information")
print(df.info())

print("\nStatistical Summary")
print(df.describe())

print("\nColumns in dataset:")
print(df.columns)


Missing Values
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
dtype: int64

Dataset Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
dtypes: float64(1), int64(9)
memory usage: 23.8 KB
None

Statistical Summary
              age         sex          cp    trestbps        chol         fbs  \
count  303.000000  303.000000  303.000000  303.000000  303.000000  303.000000  

In [5]:
df['target'] = df['exang']
print("\nCreated 'target' from 'exang'. Values:", sorted(df['target'].unique()))


Created 'target' from 'exang'. Values: [np.int64(0), np.int64(1)]


In [6]:
if df['age'].nunique() > 10:
    df['age'] = pd.cut(df['age'], bins=5, labels=False)

if df['chol'].nunique() > 10:
    df['chol'] = pd.cut(df['chol'], bins=5, labels=False)


In [7]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.parameter_estimator import DiscreteMLE


In [8]:
model = DiscreteBayesianNetwork([

    ('age', 'target'),
    ('sex', 'target'),
    ('cp', 'target'),
    ('exang', 'target'),
    ('target', 'restecg'),
    ('target', 'chol')

])


In [9]:
model.fit(df, estimator=DiscreteMLE())

print("\nBayesian Network Created Successfully")



Bayesian Network Created Successfully


In [10]:
print("\nCPD of AGE\n")
print(model.get_cpds('age'))

print("\nCPD of SEX\n")
print(model.get_cpds('sex'))

print("\nCPD of EXANG\n")
print(model.get_cpds('exang'))

print("\nCPD of TARGET\n")
print(model.get_cpds('target'))

print("\nCPD of CHOLESTEROL\n")
print(model.get_cpds('chol'))



CPD of AGE

+--------+-----------+
| age(0) | 0.039604  |
+--------+-----------+
| age(1) | 0.234323  |
+--------+-----------+
| age(2) | 0.320132  |
+--------+-----------+
| age(3) | 0.349835  |
+--------+-----------+
| age(4) | 0.0561056 |
+--------+-----------+

CPD of SEX

+--------+----------+
| sex(0) | 0.316832 |
+--------+----------+
| sex(1) | 0.683168 |
+--------+----------+

CPD of EXANG

+----------+----------+
| exang(0) | 0.673267 |
+----------+----------+
| exang(1) | 0.326733 |
+----------+----------+

CPD of TARGET

+-----------+----------+----------+----------+----------+-----+----------+----------+----------+----------+----------+
| age       | age(0)   | age(0)   | age(0)   | age(0)   | ... | age(4)   | age(4)   | age(4)   | age(4)   | age(4)   |
+-----------+----------+----------+----------+----------+-----+----------+----------+----------+----------+----------+
| cp        | cp(0)    | cp(0)    | cp(0)    | cp(0)    | ... | cp(2)    | cp(3)    | cp(3)    | cp(3) 

In [11]:
print("\nModel Check")
print(model.check_model())



Model Check
True


In [12]:
print("\nNodes")
print(model.nodes())

print("\nEdges")
print(model.edges())



Nodes
['age', 'target', 'sex', 'cp', 'exang', 'restecg', 'chol']

Edges
[('age', 'target'), ('target', 'restecg'), ('target', 'chol'), ('sex', 'target'), ('cp', 'target'), ('exang', 'target')]
